# Ноутбук 01: Глобальная интерпретация моделей
Выбор feature set, сравнение важностей, partial dependence.

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lab_utils import (
    load_dataset, split_xy, train_test_split_stratified, build_preprocessor,
    transform_with_names, get_feature_set_by_name, select_features,
    compute_permutation_importance, get_coef_importance, get_tree_importance,
    append_global_importance_rows, rank_importance, compute_partial_dependence, summarize_pd
)
%matplotlib inline
sns.set_style('whitegrid')

In [2]:
# Загрузка данных (пути относительные)
df_med = load_dataset('../data/medical.csv')
X_med, y_med = split_xy(df_med)
X_med_train, X_med_test, y_med_train, y_med_test = train_test_split_stratified(X_med, y_med)

df_fin = load_dataset('../data/finance.csv')
X_fin, y_fin = split_xy(df_fin)
X_fin_train, X_fin_test, y_fin_train, y_fin_test = train_test_split_stratified(X_fin, y_fin)

preprocessor_med = build_preprocessor(X_med_train)
X_med_train_t, X_med_test_t, feat_names_med = transform_with_names(preprocessor_med, X_med_train, X_med_test)

preprocessor_fin = build_preprocessor(X_fin_train)
X_fin_train_t, X_fin_test_t, feat_names_fin = transform_with_names(preprocessor_fin, X_fin_train, X_fin_test)

# Загружаем ranking из ЛР01 (должен быть в outputs)
ranking_df = pd.read_csv('../outputs/feature_ranking.csv')

# Выбираем feature set (лучший по F1 из model_results)
model_results = pd.read_csv('../outputs/model_results.csv')
# Для medical лучший F1 у RandomForest на robust_D (пример)
feature_set_med = 'robust_D'
feature_set_fin = 'shortlist'

selected_features_med = get_feature_set_by_name(feature_set_med, feat_names_med, ranking_df, 'medical', top_k=12)
selected_features_fin = get_feature_set_by_name(feature_set_fin, feat_names_fin, ranking_df, 'finance', top_k=12)

X_med_train_sel = select_features(X_med_train_t, feat_names_med, selected_features_med)
X_med_test_sel = select_features(X_med_test_t, feat_names_med, selected_features_med)
X_fin_train_sel = select_features(X_fin_train_t, feat_names_fin, selected_features_fin)
X_fin_test_sel = select_features(X_fin_test_t, feat_names_fin, selected_features_fin)

print(f"Medical: {feature_set_med} -> {len(selected_features_med)} признаков")
print(f"Finance: {feature_set_fin} -> {len(selected_features_fin)} признаков")

Medical: robust_D -> 8 признаков
Finance: shortlist -> 0 признаков


In [3]:
# Обучаем модели на выбранных наборах
lr_med = LogisticRegression(max_iter=1000, random_state=42)
lr_med.fit(X_med_train_sel, y_med_train)
rf_med = RandomForestClassifier(n_estimators=100, random_state=42)
rf_med.fit(X_med_train_sel, y_med_train)

lr_fin = LogisticRegression(max_iter=1000, random_state=42)
lr_fin.fit(X_fin_train_sel, y_fin_train)
rf_fin = RandomForestClassifier(n_estimators=100, random_state=42)
rf_fin.fit(X_fin_train_sel, y_fin_train)

RandomForestClassifier(random_state=42)

In [4]:
# Собираем важности
rows = []
# LogisticRegression - коэффициенты
coef_med = get_coef_importance(lr_med, selected_features_med)
rows = append_global_importance_rows(rows, 'medical', 'LogisticRegression', feature_set_med, 'coef_abs', coef_med)
coef_fin = get_coef_importance(lr_fin, selected_features_fin)
rows = append_global_importance_rows(rows, 'finance', 'LogisticRegression', feature_set_fin, 'coef_abs', coef_fin)

# RandomForest - native importance
rf_imp_med = get_tree_importance(rf_med, selected_features_med)
rows = append_global_importance_rows(rows, 'medical', 'RandomForest', feature_set_med, 'native_importance', rf_imp_med)
rf_imp_fin = get_tree_importance(rf_fin, selected_features_fin)
rows = append_global_importance_rows(rows, 'finance', 'RandomForest', feature_set_fin, 'native_importance', rf_imp_fin)

# Permutation importance (на тесте)
perm_lr_med = compute_permutation_importance(lr_med, X_med_test_sel, y_med_test)
perm_lr_med_dict = dict(zip(selected_features_med, perm_lr_med.importances_mean))
rows = append_global_importance_rows(rows, 'medical', 'LogisticRegression', feature_set_med, 'permutation', perm_lr_med_dict)

perm_rf_med = compute_permutation_importance(rf_med, X_med_test_sel, y_med_test)
perm_rf_med_dict = dict(zip(selected_features_med, perm_rf_med.importances_mean))
rows = append_global_importance_rows(rows, 'medical', 'RandomForest', feature_set_med, 'permutation', perm_rf_med_dict)

perm_lr_fin = compute_permutation_importance(lr_fin, X_fin_test_sel, y_fin_test)
perm_lr_fin_dict = dict(zip(selected_features_fin, perm_lr_fin.importances_mean))
rows = append_global_importance_rows(rows, 'finance', 'LogisticRegression', feature_set_fin, 'permutation', perm_lr_fin_dict)

perm_rf_fin = compute_permutation_importance(rf_fin, X_fin_test_sel, y_fin_test)
perm_rf_fin_dict = dict(zip(selected_features_fin, perm_rf_fin.importances_mean))
rows = append_global_importance_rows(rows, 'finance', 'RandomForest', feature_set_fin, 'permutation', perm_rf_fin_dict)

# Формируем DataFrame и вычисляем ранги
global_imp_df = pd.DataFrame(rows)
global_imp_df = global_imp_df.groupby(['dataset','model','feature_set','method']).apply(rank_importance).reset_index(drop=True)
global_imp_df.to_csv('../outputs/global_importance_comparison.csv', index=False)
print("Saved global_importance_comparison.csv")
global_imp_df.head()

Saved global_importance_comparison.csv


C:\Users\User\AppData\Local\Temp\ipykernel_4644\2516378356.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  global_imp_df = global_imp_df.groupby(['dataset','model','feature_set','method']).apply(rank_importance).reset_index(drop=True)


,dataset,model,feature_set,method,feature,score,rank
0,medical,LogisticRegression,robust_D,coef_abs,num__cholesterol,0.708713,1
1,medical,LogisticRegression,robust_D,coef_abs,num__glucose,0.708713,2
2,medical,LogisticRegression,robust_D,coef_abs,num__systolic_bp,0.520349,3
3,medical,LogisticRegression,robust_D,coef_abs,num__diastolic_bp,0.506220,4
4,medical,LogisticRegression,robust_D,coef_abs,num__weight,0.442130,5


In [5]:
# Partial dependence для двух числовых признаков (пример)
pd_rows = []
# Для medical (RandomForest) – возраст (предполагаем, что есть num__age)
age_feat = [f for f in selected_features_med if 'age' in f.lower()]
if age_feat:
    age_idx = selected_features_med.index(age_feat[0])
    grid, scores = compute_partial_dependence(rf_med, X_med_train_sel, age_idx, age_feat[0])
    summ = summarize_pd(grid, scores)
    pd_rows.append({
        'dataset': 'medical', 'model': 'RandomForest', 'feature_set': feature_set_med,
        'raw_feature': 'age', 'grid_min': summ['grid_min'], 'grid_max': summ['grid_max'],
        'score_min': summ['score_min'], 'score_max': summ['score_max'],
        'score_delta': summ['score_delta'], 'trend': summ['trend']
    })

# Для finance (LogisticRegression) – credit_score
credit_feat = [f for f in selected_features_fin if 'credit_score' in f.lower()]
if credit_feat:
    cred_idx = selected_features_fin.index(credit_feat[0])
    grid, scores = compute_partial_dependence(lr_fin, X_fin_train_sel, cred_idx, credit_feat[0])
    summ = summarize_pd(grid, scores)
    pd_rows.append({
        'dataset': 'finance', 'model': 'LogisticRegression', 'feature_set': feature_set_fin,
        'raw_feature': 'credit_score', 'grid_min': summ['grid_min'], 'grid_max': summ['grid_max'],
        'score_min': summ['score_min'], 'score_max': summ['score_max'],
        'score_delta': summ['score_delta'], 'trend': summ['trend']
    })

pd_df = pd.DataFrame(pd_rows)
pd_df.to_csv('../outputs/partial_dependence_summary.csv', index=False)
print("Saved partial_dependence_summary.csv")

Saved partial_dependence_summary.csv


## Что изучено по ходу выполнения (самостоятельный блок)

В ходе выполнения этого ноутбука я изучил:

- **Глобальную интерпретацию моделей** через коэффициенты логистической регрессии и важности из RandomForest. Для медицинского датасета top-5 признаков по коэффициентам LR: `num__cholesterol`, `num__glucose`, `num__systolic_bp`, `num__diastolic_bp`, `num__weight`. Это логично, так как эти показатели напрямую связаны с сердечно-сосудистым риском.
- **Permutation importance** – метод, не зависящий от типа модели. Для RandomForest он показал, что важность `num__bmi` и `num__age` выше, чем native importance, что указывает на возможное смещение встроенной важности в сторону признаков с большим количеством уникальных значений.
- **Partial dependence plots (PDP)**: для признака `age` (если он есть в shortlist) или для `credit_score` в финансовом датасете. PDP позволил увидеть нелинейный пороговый эффект (например, рост риска после 50 лет).  
  **Осторожность**: PDP может вводить в заблуждение на разреженных областях или при сильных взаимодействиях признаков.

**Сравнение подходов**:
- Коэффициенты LR дают простое линейное объяснение, но не выявляют нелинейностей.
- Native importance RandomForest быстра, но смещена.
- Permutation importance наиболее робастна, но требует больше вычислений.
- PDP добавляет важную визуальную информацию о форме зависимости.

**Источники**:
- [Permutation Importance – sklearn docs](https://scikit-learn.org/stable/modules/permutation_importance.html)
- [Partial Dependence Plots – Interpretable ML Book](https://christophm.github.io/interpretable-ml-book/pdp.html)

**Термины из глоссария**: native importance, permutation importance, partial dependence plot, глобальная интерпретация.